In [1]:
import os
import subprocess
import time

# 0. Cài đặt zstd cho Colab (Fix lỗi giải nén Ollama)
!apt-get update -y -q
!apt-get install -y -q zstd

# Dọn dẹp các tiến trình kẹt cũ
os.system("pkill ollama")
os.system("pkill ngrok")

# 1. Cài đặt các thư viện cần thiết
!pip install pyngrok huggingface_hub -q

from pyngrok import ngrok
from huggingface_hub import hf_hub_download

ngrok.kill() # Xóa các đường hầm cũ

# 2. Cài đặt phần mềm Ollama vào Colab
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Khởi động Ollama
os.environ["OLLAMA_HOST"] = "0.0.0.0"
print("Khởi động Ollama server...")
subprocess.Popen("ollama serve", shell=True)
time.sleep(5)

# 4. Tải file GGUF 4.5GB từ HuggingFace
print("Đang tải model từ HuggingFace...")
hf_hub_download(repo_id="xunnhi/Qwen2.5-7B-RAG-LoRA", filename="qwen2.5-7b-instruct.Q4_K_M.gguf", local_dir=".")

# 5. Tạo file cấu hình Modelfile
with open("Modelfile", "w") as f:
    f.write('FROM ./qwen2.5-7b-instruct.Q4_K_M.gguf\nTEMPLATE """<|im_start|>system\n{{ .System }}<|im_end|>\n<|im_start|>user\n{{ .Prompt }}<|im_end|>\n<|im_start|>assistant\n"""\nPARAMETER temperature 0.1\nPARAMETER num_ctx 4096')

# 6. Đóng gói model vào Ollama
print("Đang cài đặt model vào Ollama...")
os.system("ollama create qwen2.5-rag -f Modelfile")

# ==========================================
# 7. CHÈN MÃ TOKEN NGROK CỦA BẠN VÀO ĐÂY:
ngrok.set_auth_token("2xBVYeSqv5NOm3jZqmeoi8KrnkD_4njEbVZvbdDnyQk7sW4im")
# ==========================================

# 8. Mở cổng kết nối Internet
public_url = ngrok.connect(11434).public_url
print(f"OLLAMA API URL CỦA BẠN LÀ: {public_url}/v1")


Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [355 B]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,719 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,053 kB]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


qwen2.5-7b-instruct.Q4_K_M.gguf:   0%|          | 0.00/4.68G [00:00<?, ?B/s]

Đang cài đặt model vào Ollama...
OLLAMA API URL CỦA BẠN LÀ: https://336c-136-109-236-14.ngrok-free.app/v1
